# Ad Impressions Forecasting

## Project Overview

This project forecasts **daily ad impressions** over a 14-day horizon.
We generate synthetic daily impression data for an ad platform spanning 2 years
with weekly seasonality, quarterly budget cycles, and holiday seasonality.

**Models**: Naive baselines, LazyPredict (tabular), FLAML AutoML, StatsForecast.
**Forecast horizon**: 14 days ahead.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily ad impression counts, predict the next 14 days. Accurate forecasts help ad platforms plan inventory, set pricing (CPM), and meet advertiser commitments.

## Why This Project Matters

Ad impression forecasting is critical for digital advertising revenue. Under-forecasting means selling impressions at discount (programmatic fill); over-forecasting means unmet guarantees to direct advertisers.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily ad impressions
- Weekly seasonality (lower weekends)
- Growth trend
- Quarterly budget flush patterns (end-of-quarter spikes)
- Holiday dips (Christmas week)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `impressions` | Daily ad impression count (millions) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'impressions'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}, freq={FREQ}")

Config: 730 points, horizon=14, season=7, freq=D


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

trend = 50 + 0.03 * t  # millions
weekly = 8 * np.sin(2 * np.pi * (t + 1) / 7)  # weekday higher

# Quarterly budget flush
quarterly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if m in [3, 6, 9, 12] and d >= 25:
        quarterly[i] = 15

# Holiday dip
holiday_dip = np.zeros(N_POINTS)
for i in range(N_POINTS):
    if dates[i].month == 12 and dates[i].day >= 24:
        holiday_dip[i] = -20

noise = np.random.normal(0, 4, N_POINTS)
impressions = trend + weekly + quarterly + holiday_dip + noise
impressions = np.maximum(impressions, 5).round(1)

df = pd.DataFrame({'date': dates, 'impressions': impressions})
print(f"Dataset shape: {df.shape}")
print(f"Unit: millions of impressions")
df.head()

Dataset shape: (730, 2)
Unit: millions of impressions


,date,impressions
0,2022-01-01,58.2
1,2022-01-02,57.3
2,2022-01-03,56.1
3,2022-01-04,52.7
4,2022-01-05,41.4


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('Ad Impressions Forecasting Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rolling_w = min(SEASON_LENGTH * 2, N_POINTS // 4)
df['_rolling'] = df[TARGET].rolling(rolling_w).mean()
axes[1, 1].plot(df['date'], df['_rolling'])
axes[1, 1].set_title(f'Rolling {rolling_w}-period Mean')
df.drop(columns='_rolling', inplace=True)
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA plots saved.")

EDA plots saved.


## Target Analysis

In [7]:
print("impressions Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

impressions Statistics:
count    730.000000
mean      61.528767
std        9.825396
min       33.900000
25%       54.700000
50%       61.700000
75%       68.400000
max       92.300000
Name: impressions, dtype: float64

CV: 0.160
Skewness: 0.092


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all points except last 28
- **Validation**: next 14
- **Test**: last 14

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree-based models handle raw features well. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek if 'D' in ['D','h','H'] else 0
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day if 'D' in ['D','h','H'] else 1
    if 'D' in ['h', 'H']:
        d['hour'] = d['date'].dt.hour
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 13 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:       16.2 | RMSE:       17.3 | MAPE: 25.27%
Seasonal Naive (7)             | MAE:        5.9 | RMSE:        9.8 | MAPE:  9.52%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
Empty DataFrame
Columns: []
Index: []


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        4.2 | RMSE:        5.5 | MAPE:  5.80%
FLAML Test (catboost)          | MAE:        4.5 | RMSE:        7.1 | MAPE:  7.31%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]

sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))

print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        5.7 | RMSE:        8.6 | MAPE:  9.25%
SF AutoETS                     | MAE:        5.9 | RMSE:        8.8 | MAPE:  9.56%
SF SeasonalNaive               | MAE:        6.4 | RMSE:        9.6 | MAPE: 10.11%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model       MAE      RMSE      MAPE
     FLAML (catboost)  4.232689  5.469600  5.798083
FLAML Test (catboost)  4.509541  7.125794  7.308290
         SF AutoARIMA  5.733133  8.550802  9.253905
           SF AutoETS  5.921649  8.794725  9.557464
   Seasonal Naive (7)  5.950000  9.830746  9.524423
     SF SeasonalNaive  6.385714  9.625339 10.107611
   Naive (Last Value) 16.157143 17.263297 25.273085

Top 3:
                model      MAE     RMSE     MAPE
     FLAML (catboost) 4.232689 5.469600 5.798083
FLAML Test (catboost) 4.509541 7.125794 7.308290
         SF AutoARIMA 5.733133 8.550802 9.253905


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -2.95, Std: 6.48


## Interpretation and Business Insight

- Ad impressions follow a clear weekday > weekend pattern
- End-of-quarter budget flush creates predictable 20-30% spikes
- Christmas week shows significant dips as advertisers pause campaigns
- Growth trend reflects increasing platform user base
- FLAML with lag features captures the weekly cycle effectively

## Limitations

1. Synthetic data — real ad data depends on advertiser budgets and bidding
2. Single platform aggregate — real platforms segment by ad unit, geo, device
3. No advertiser-side features (budgets, campaigns, targeting changes)
4. Programmatic vs. direct sold not differentiated

## How to Improve This Project

1. Add advertiser campaign schedule as features
2. Segment by ad format (display, video, native)
3. Incorporate user growth forecasts as an input
4. Model fill rate alongside impression volume
5. Use probabilistic forecasts for guaranteed vs. programmatic split

## Production Considerations

- Daily forecasts feeding into yield management systems
- Integration with ad server pacing algorithms
- Alerting on forecast deviations > 10% from actual
- Quarterly re-calibration for budget cycle changes

## Common Mistakes

1. Ignoring quarterly budget cycles in ad data
2. Not handling holiday dips separately from regular seasonality
3. Treating weekends the same as weekdays
4. Using raw impressions without accounting for viewability

## Mini Challenge / Exercises

1. Add a quarterly dummy feature and measure impact
2. Forecast separately for weekdays and weekends
3. Simulate a new advertiser joining — how does it affect forecasts?
4. Build a fill-rate model on top of impression forecasts

## Final Summary / Key Takeaways

1. Ad impressions have strong weekly and quarterly patterns
2. Holiday periods require special handling (dips, not spikes)
3. End-of-quarter budget flushes are predictable and should be features
4. 14-day forecasts align well with ad campaign planning cycles
5. Combining FLAML with AutoARIMA provides robust forecasts

In [18]:
import json
metrics = {
    'project': 'Ad Impressions Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Ad Impressions Forecasting — Complete ===")

Metrics saved.

=== Ad Impressions Forecasting — Complete ===
